# Experimento 4: más datos (The Call of Cthulhu en Train)
Basado en la versión multi-cuento de la Etapa 1 que armó Brian. Se suma *The Call of Cthulhu* (Project Gutenberg #68283, en `db/`) **solo a Train**; Dev sigue siendo el de *The Festival*. Se usan solo las ventanas cuyos caracteres están en el vocabulario de *The Festival* (67.212 de 69.810). R4 con 3 semillas, medido solo en Dev (Test ya se usó en la Etapa 5).

| Train | Ventanas | Dev | Perplejidad |
|---|---|---|---|
| Solo *The Festival* | 12.780 | 50,31 % ± 0,48 % | 5,89 |
| *The Festival* + *Cthulhu* | 79.992 | **54,40 % ± 0,58 %** | 4,92 |

**Resultado:** +4,09 pp en Dev, más que cualquier cambio de arquitectura del trabajo. El límite eran los datos.


In [2]:
# ==============================================================================
# 1.1 CONFIGURACIÓN DEL ENTORNO
# ==============================================================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import json
import time
import urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

SEED = 42
SEEDS = [42, 123, 456, 789, 2026]
np.random.seed(SEED)

print(f"TensorFlow {tf.__version__} | Keras {keras.__version__}")

TensorFlow 2.21.0 | Keras 3.15.1


In [3]:
# ==============================================================================
# 1.2 CARGA DEL TEXTO LIMPIO Y DE LA PARTICIÓN DE LA ETAPA 1
# ==============================================================================
BASE_DIR = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
PROC = BASE_DIR / "data" / "processed"

text = (PROC / "The_Festival_clean.txt").read_text(encoding="utf-8")
splits = json.load(open(PROC / "splits_bloques.json", encoding="utf-8"))

L = splits["L"]
limites = splits["limites_bloques"]
bloques_split = splits["bloques"]
vocabulary = list(splits["vocabulario"])
char2idx = {c: i for i, c in enumerate(vocabulary)}
idx2char = {i: c for c, i in char2idx.items()}
# Muestra el espacio y el salto de línea de forma legible en tablas y gráficos
vis = lambda c: {" ": "<SP>", "\n": "<NL>"}.get(c, c)
V = len(vocabulary)

def ventanas_de_bloques(ids):
    X, y = [], []
    for b in ids:
        ini, fin = limites[b]
        for i in range(ini, fin - L):
            X.append(text[i:i+L])
            y.append(text[i+L])
    return X, y

X_train, y_train = ventanas_de_bloques(bloques_split["train"])
X_dev,   y_dev   = ventanas_de_bloques(bloques_split["dev"])

# Entrada como índices enteros: (N, L)
a_indices = lambda X: np.array([[char2idx[c] for c in x] for x in X], dtype=np.int32)
Xi_train, Xi_dev = a_indices(X_train), a_indices(X_dev)
yi_train = np.array([char2idx[c] for c in y_train], dtype=np.int32)
yi_dev   = np.array([char2idx[c] for c in y_dev], dtype=np.int32)

print(f"|V| = {V} | L = {L}")
print(f"Train: {len(X_train):,} muestras | Dev: {len(X_dev):,} muestras")

|V| = 64 | L = 40
Train: 12,780 muestras | Dev: 2,880 muestras


In [4]:
# ==============================================================================
# 1.3 REFERENCIAS DE LAS ETAPAS 1 Y 2
# ==============================================================================
ref_base = json.load(open(PROC / "baseline_dev.json", encoding="utf-8"))
reg2 = json.load(open(PROC / "resultados_modelos.json", encoding="utf-8"))
ref2 = reg2["configuraciones"][reg2["ganador"]]

# Configuración de entrenamiento heredada del ganador de la Etapa 2
CFG2 = ref2["config"]
LR = CFG2["lr"]
DROPOUT = CFG2.get("dropout", 0.0)
EPOCAS = reg2["learning_rate"]["epocas"]
BATCH = 128
K = ref_base["K"]

print(f"Baseline (Etapa 1): {ref_base['accuracy_dev']*100:.2f}% en Dev")
print(f"Ganador Etapa 2 ({reg2['ganador']}): {ref2['dev_acc_mean']*100:.2f}% ± {ref2['dev_acc_std']*100:.2f}% en Dev")
print(f"Configuración heredada: lr = {LR:g} | Dropout = {DROPOUT} | {EPOCAS} épocas | batch {BATCH} | K = {K}")

Baseline (Etapa 1): 40.83% en Dev
Ganador Etapa 2 (Dropout 0.05): 48.71% ± 0.52% en Dev
Configuración heredada: lr = 0.0005 | Dropout = 0.05 | 100 épocas | batch 128 | K = 5


In [5]:
# ==============================================================================
# 2.1 CONSTRUCCIÓN, ENTRENAMIENTO Y EVALUACIÓN CON 5 SEMILLAS
# ==============================================================================
def construir_modelo(tipo, k=K, emb_dim=16, emb_init=None, emb_trainable=True, dropout=DROPOUT, unidades_lstm=128):
    """
    tipo = "onehot": MLP de la Etapa 2 sobre one-hot aplanado.
    tipo = "densa":  Embedding -> Flatten -> MLP 128-64-32.
    tipo = "lstm":   Embedding -> LSTM -> Dropout -> softmax.
    """
    entrada = layers.Input(shape=(k,), dtype="int32")
    if tipo == "onehot":
        # One-hot exacto: embedding fijo con la matriz identidad
        x = layers.Embedding(V, V, embeddings_initializer=keras.initializers.Identity(), trainable=False)(entrada)
    else:
        init = keras.initializers.Constant(emb_init) if emb_init is not None else "uniform"
        x = layers.Embedding(V, emb_dim, embeddings_initializer=init, trainable=emb_trainable, name="embedding")(entrada)

    if tipo in ("onehot", "densa"):
        x = layers.Flatten()(x)
        for u in [128, 64, 32]:
            x = layers.Dense(u, activation="relu")(x)
            if dropout > 0:
                x = layers.Dropout(dropout)(x)
    else:
        x = layers.LSTM(unidades_lstm)(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)

    salida = layers.Dense(V, activation="softmax")(x)
    return keras.Model(entrada, salida)

def evaluar(constructor, k, epochs=EPOCAS, seeds=SEEDS, lr=None, clipnorm=None, use_es=False, patience=5):
    """Entrena con cada semilla y devuelve (resumen, historial y modelo de la semilla 42)."""
    Xtr, Xdv = Xi_train[:, -k:], Xi_dev[:, -k:]
    corridas = []
    for s in seeds:
        keras.utils.set_random_seed(s)
        modelo = constructor()
        modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr or LR, clipnorm=clipnorm),
                       loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        cbs = []
        if use_es:
            cbs.append(callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True))
        t0 = time.perf_counter()
        h = modelo.fit(Xtr, yi_train, validation_data=(Xdv, yi_dev), epochs=epochs, batch_size=BATCH,
                       callbacks=cbs, verbose=0)
        t_run = time.perf_counter() - t0
        if s == SEED:
            h42, m42 = h.history, modelo
        loss_tr, acc_tr = [float(v) for v in modelo.evaluate(Xtr, yi_train, verbose=0)]
        loss_dv, acc_dv = [float(v) for v in modelo.evaluate(Xdv, yi_dev, verbose=0)]
        va = h.history["val_accuracy"]
        corridas.append({"seed": s, "tiempo": t_run, "train_acc": acc_tr, "dev_acc": acc_dv,
                         "brecha": acc_tr - acc_dv, "dev_loss": loss_dv, "dev_perp": float(np.exp(loss_dv)),
                         "pico_acc": float(max(va)), "ep_pico_acc": int(np.argmax(va)) + 1,
                         "epocas": len(h.history["loss"])})
    df_c = pd.DataFrame(corridas)
    resumen = {"params": int(sum(np.prod(w.shape) for w in m42.trainable_weights)), "k": k, "dev_acc_min": float(df_c["dev_acc"].min()), "corridas": corridas}
    for col in df_c.columns.drop("seed"):
        resumen[f"{col}_mean"] = float(df_c[col].mean())
        resumen[f"{col}_std"] = float(df_c[col].std())
    return resumen, h42, m42



In [6]:
# Cthulhu: misma limpieza que la Etapa 1 y solo ventanas cuyos 41 caracteres estén en el vocabulario de The Festival
import re
raw_c = open(BASE_DIR / "db" / "The_Call_of_Cthulhu.txt", encoding="utf-8").read()
# Edición de Project Gutenberg (ebook 68283): se quitan la cabecera, la licencia, la nota del transcriptor,
# el epígrafe de la ilustración y las marcas de cursiva (_texto_); la nota al pie de Lovecraft queda como texto
raw_c = raw_c.split("*** START OF THE PROJECT GUTENBERG EBOOK 68283 ***")[1].split("*** END OF THE PROJECT GUTENBERG EBOOK 68283 ***")[0]
raw_c = re.sub(r"\[Transcriber's Note:.*?\]", "", raw_c, flags=re.S)
raw_c = re.sub(r"\[Illustration:.*?\]\[1\]", "", raw_c, flags=re.S)
raw_c = re.sub(r"\[Footnote 1: (.*?)\]", r"\1", raw_c, flags=re.S)
raw_c = raw_c.replace("_", "")
for fn in [lambda t: t.replace("\r\n", "\n").replace("\r", "\n"),
           lambda t: re.sub(r"</?i>", "", t),
           lambda t: t.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"'),
           lambda t: re.sub(r"\s+", " ", t).strip()]:
    raw_c = fn(raw_c)
ok = set(vocabulary)
Xc = [raw_c[i:i+L] for i in range(len(raw_c) - L) if set(raw_c[i:i+L+1]) <= ok]
yc = [raw_c[i+L] for i in range(len(raw_c) - L) if set(raw_c[i:i+L+1]) <= ok]
print(f"Cthulhu limpio: {len(raw_c):,} caracteres | ventanas: {len(raw_c) - L:,} | con vocabulario de The Festival: {len(Xc):,}")
print(f"Caracteres de Cthulhu fuera del vocabulario: {sorted(set(raw_c) - ok)}")
print(f"Inicio: {raw_c[:120]!r}")
print(f"Final:  {raw_c[-80:]!r}")
Xi_c = a_indices(Xc)
yi_c = np.array([char2idx[ch] for ch in yc], dtype=np.int32)

Cthulhu limpio: 69,850 caracteres | ventanas: 69,810 | con vocabulario de The Festival: 67,212


Caracteres de Cthulhu fuera del vocabulario: ['!', '&', '(', ')', '*', '3', '4', '7', '?', 'Q', 'V', 'Z', '°', '´', 'é']
Inicio: 'The CALL of CTHULHU By H.P. LOVECRAFT "Of such great powers or beings there may be conceivably a survival ... a survival'
Final:  'my executors may put caution before audacity and see that it meets no other eye.'


In [7]:
import json
SEEDS3 = [42, 123, 456]
cfg = dict(lr=LR, clipnorm=1.0, use_es=True, patience=5)
salida = {}
for nombre, (Xtr_full, ytr) in {"R4 — solo The Festival": (Xi_train, yi_train),
                                "R4 — The Festival + Cthulhu en Train": (np.concatenate([Xi_train, Xi_c]), np.concatenate([yi_train, yi_c]))}.items():
    Xi_train_bak, yi_train_bak = Xi_train, yi_train
    Xi_train, yi_train = Xtr_full, ytr          # evaluar() lee Xi_train / yi_train
    r, h, _ = evaluar(lambda: construir_modelo("lstm", k=K, emb_dim=300), k=K, seeds=SEEDS3, **cfg)
    Xi_train, yi_train = Xi_train_bak, yi_train_bak
    salida[nombre] = {k: v for k, v in r.items() if k != "corridas"} | {"n_train": len(ytr),
        "dev_por_semilla": [x["dev_acc"] for x in r["corridas"]], "ep_por_semilla": [x["epocas"] for x in r["corridas"]]}
    print(f"{nombre:<38} Train {len(ytr):>6,} | Dev {r['dev_acc_mean']*100:.2f}% ± {r['dev_acc_std']*100:.2f}% "
          f"{[round(x['dev_acc']*100, 2) for x in r['corridas']]} | Train acc {r['train_acc_mean']*100:.2f}% | "
          f"perp {r['dev_perp_mean']:.2f} | {r['epocas_mean']:.0f} épocas | {r['tiempo_mean']:.0f} s", flush=True)

R4 — solo The Festival                 Train 12,780 | Dev 50.31% ± 0.48% [50.0, 50.07, 50.87] | Train acc 60.29% | perp 5.89 | 46 épocas | 46 s


R4 — The Festival + Cthulhu en Train   Train 79,992 | Dev 54.40% ± 0.58% [53.75, 54.86, 54.58] | Train acc 61.33% | perp 4.92 | 43 épocas | 287 s
